# Day-1 realistic LLM experiment: Colab A100 runner (GATED)

One-click pipeline for a REALISTIC version of the `llm/` conflicting-preference
experiment: Qwen2.5-0.5B-Instruct + LoRA, TRL `GRPOTrainer`, on real **GSM8K** math
word problems instead of a synthetic integer-picking task. Every training prompt is
randomly assigned (50/50, hidden from the model) to one of two REAL graders that
disagree about HOW to answer correctly, not WHETHER:

- **Grader A, "reasoning rubric" (0-10)**: `10 * correct * min(1, n_steps/6)` --
  rewards a correct final answer that shows worked reasoning, up to 6 steps.
- **Grader B, "terse verifier" (0-1, SMOOTH)**: `correct * max(0, 1 - n_tokens/400)` --
  rewards a correct final answer that tapers LINEARLY to 0 by 400 tokens, not a hard
  cliff (v1's hard `<=40 tokens` cliff meant essentially no real-model completion ever
  qualified, leaving grader B with zero reward AND zero signal everywhere).

Final-answer parsing falls back through three tiers: `#### <number>`, then
`\boxed{<number>}`, then the LAST number anywhere in the completion (v1's strict
`#### only` parse left most completions unparsable, starving both graders).
Correctness is shared; verbosity conflicts. The two graders' NATIVE scales (0-10 vs
0-1) are themselves a realistic version of `llm/`'s artificial per-family reward-scale
`k` -- there is no synthetic scaling knob here.

**v2**: a HARD baseline gate -- Step 1 must show accuracy >= 20% AND both graders'
true within-GROUP reward std > 0 before Step 2 is allowed to spend any GPU time.

**v3 (attempt 4)**: v2's 200-token completion cap turned out to BE the problem --
completions were getting truncated at the cap (median length = 200), which hurt
accuracy and left grader B's reward near 0 (`1 - 200/200 ~ 0`). Raised the cap to 320
and grader B's taper denominator to 400. Step 1 also now tries
`config.MODEL_NAME` (Qwen2.5-0.5B-Instruct) first and AUTOMATICALLY retries with
`config.MODEL_NAME_FALLBACK` (Qwen2.5-1.5B-Instruct) if the small model can't clear
the accuracy gate. Every attempt (pass or fail, baseline or pilot) is appended to
`attempts.json` on Drive -- the one file that survives the results wipe below, so the
full history of what was tried stays honest.

**This notebook is GATED: it STOPS after the Step 2 pilot gate.** No full sweep yet.
All results are saved to Google Drive under `grpo_llm_real_results/` -- a SEPARATE
folder from `llm/`'s `grpo_llm_results/`, so this never touches `llm/`'s results.

## 1. Check for a GPU

Sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` before torch ever loads.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess

try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False

if not has_gpu:
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type, select a GPU '
        '(A100 recommended), then Runtime > Restart session and run all cells again.'
    )

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2. Config -- edit if your fork/repo name differs

In [ ]:
GITHUB_USER = "nilay47"
GITHUB_REPO = "agenthon"
# Deliberately a DIFFERENT folder from llm/'s "grpo_llm_results" -- this experiment
# must never read or write llm/'s results.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/grpo_llm_real_results"
REPO_DIR = f"/content/{GITHUB_REPO}"

## 3. Mount Google Drive

Shares the SAME HuggingFace cache location as `llm/`'s notebook (just a model/dataset
download cache, not results) so the Qwen2.5-0.5B weights don't need a second download.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to {DRIVE_RESULTS_DIR}")

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
hf_home = os.environ['HF_HOME']
print(f"HF cache: {hf_home}")

## 3b. Delete old results (keep attempts.json)

Each redesign (reward function, parsing logic, prompt template, training size, model)
makes every number in an OLDER result file stale. Wipes everything in
`DRIVE_RESULTS_DIR` (preflight files, baseline check, run results, pilot summary)
**except `attempts.json`**, which is the append-only log of every attempt tried so
far (config + prediction + outcome, pass or fail) -- the one file meant to survive
every redesign, so a failed attempt stays reported instead of silently disappearing.

In [ ]:
import shutil

if os.path.exists(DRIVE_RESULTS_DIR):
    for _name in os.listdir(DRIVE_RESULTS_DIR):
        if _name == 'attempts.json':
            continue
        _path = os.path.join(DRIVE_RESULTS_DIR, _name)
        print('deleting (stale):', _path)
        shutil.rmtree(_path) if os.path.isdir(_path) else os.remove(_path)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"{DRIVE_RESULTS_DIR} wiped (attempts.json preserved if it existed)")

## 4. Clone the repo

If the repo is **private**, add a `GITHUB_TOKEN` Colab secret first (key icon in the
left sidebar) with a personal access token that has `repo` scope.

In [ ]:
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone (pulling latest instead)")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/llm_real

## 5. Install dependencies (Colab-safe: never touches torch/CUDA)

Reuses `llm/requirements-colab.txt` as-is (same five packages -- `trl`/`peft`/
`transformers`/`accelerate`/`datasets` -- installed with `--no-deps`, so pip never
resolves their own torch dependency against Colab's preinstalled build). `datasets`
is what loads GSM8K here too, no extra dependency needed.

In [ ]:
!pip install -q --no-deps -r ../llm/requirements-colab.txt
!pip install -q huggingface_hub tokenizers safetensors regex pyyaml filelock requests \
    tqdm packaging psutil pandas pyarrow dill multiprocess fsspec xxhash scipy
!pip uninstall -y -q torchao

import importlib.metadata as _metadata

for _pkg in ['torch', 'transformers', 'trl', 'peft', 'accelerate', 'datasets']:
    try:
        _ver = _metadata.version(_pkg)
    except _metadata.PackageNotFoundError:
        _ver = 'NOT INSTALLED'
    print(f"{_pkg} == {_ver}")

import torch

print(f"torch.__version__ = {torch.__version__}")
print(f"torch.version.cuda = {torch.version.cuda}")
print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")

try:
    import trl.trainer.grpo_trainer  # noqa: F401
    from peft import LoraConfig, get_peft_model  # noqa: F401
    print("trl.trainer.grpo_trainer and peft imported OK")
except Exception as e:
    raise RuntimeError(
        f"Import check failed against torch {torch.__version__} (CUDA {torch.version.cuda}). "
        f"Original error: {e!r}"
    ) from e

## 5b. PREFLIGHT: exercise GRPO and Dr.GRPO on 2 tiny steps

Loads the real model + LoRA and runs 2 GRPO steps (2 prompts x G=4, 16-token
completions) for each of the two methods the pilot gate compares, including one eval
callback call and one save to the Drive `preflight/` subfolder each -- the exact same
code path `run.py` uses for the real runs, just small enough to finish in seconds. Also
exercises the real GSM8K load + grader-assignment path.

In [ ]:
import os
import sys
sys.path.insert(0, '.')
import config
from run import run_one

PREFLIGHT_DIR = os.path.join(DRIVE_RESULTS_DIR, 'preflight')

for _method in config.METHODS:
    print(f'--- preflight: {_method} ---')
    try:
        run_one(_method, seed=0, max_steps=2, out_dir=PREFLIGHT_DIR,
                prompts_per_step=2, g=4, max_completion_length=16, eval_every=1, n_eval_periodic=3)
    except Exception as e:
        raise RuntimeError(
            f"PREFLIGHT FAILED for config '{_method}'. Original error: {e!r}"
        ) from e

print("PREFLIGHT PASSED")

## 5c. PREFLIGHT: peak GPU memory at the REAL batch shape

The config-smoke preflight above uses a tiny 2-prompt batch and 16-token completions,
which doesn't stress memory. This runs ONE micro-batch step at the ACTUAL training
shape (`config.PROMPTS_PER_STEP` x `config.G` = 64 completions per optimizer step,
`config.MAX_COMPLETION_LENGTH`=320 tokens, with completions forced out to the full
length via `min_new_tokens` -- the true worst case) and reports peak GPU memory. Unlike
`llm/`'s 8-token task, this task keeps gradient checkpointing ON and a conservative
`config.MICRO_BATCH_COMPLETIONS` by default specifically because 320-token completions
are ~40x bigger. Fails if peak reserved memory exceeds 34 GB (A100 is 40GB) -- reduce
`config.MICRO_BATCH_COMPLETIONS` and rerun if it does. (Uses `config.MODEL_NAME`, i.e.
the PRIMARY model -- Step 1 below separately checks the fallback model's eval-time
memory if it's ever used.)

In [ ]:
import torch

torch.cuda.reset_peak_memory_stats()
run_one('grpo', seed=0, max_steps=1, out_dir=PREFLIGHT_DIR,
        prompts_per_step=config.PROMPTS_PER_STEP, g=config.G,
        max_completion_length=config.MAX_COMPLETION_LENGTH, eval_every=1, n_eval_periodic=3,
        extra_grpo_kwargs=dict(generation_kwargs=dict(min_new_tokens=config.MAX_COMPLETION_LENGTH)))

peak_allocated_gb = torch.cuda.max_memory_allocated() / 1024**3
peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f"peak memory allocated: {peak_allocated_gb:.2f} GB")
print(f"peak memory reserved:  {peak_reserved_gb:.2f} GB")

if peak_reserved_gb > 34:
    current_micro = config.MICRO_BATCH_COMPLETIONS
    raise RuntimeError(
        f"Peak GPU memory ({peak_reserved_gb:.2f} GB reserved) exceeds the 34 GB safety "
        f"threshold for a 40GB A100. Reduce config.MICRO_BATCH_COMPLETIONS (currently "
        f"{current_micro}) and rerun this cell."
    )
print("MEMORY PREFLIGHT PASSED")

## 6. STEP 1: baseline sanity check (no training) -- HARD GATE

Runs the UNTRAINED model (`config.MODEL_NAME`, Qwen2.5-0.5B-Instruct) on all 200
reserved held-out GSM8K test problems: accuracy, completion-length and n_steps
distributions, each grader's unscaled reward mean and across-problem population std,
plus the TRUE within-GROUP reward std (G samples of the SAME prompt, matching GRPO's
own group -- the quantity GRPO's advantage computation actually depends on, not the
population std).

**If it fails, automatically retries with `config.MODEL_NAME_FALLBACK`**
(Qwen2.5-1.5B-Instruct), reporting that attempt's peak GPU memory and an estimated
full pilot-run wall-clock. Saves `gate_passed` and `model_name` (whichever model, if
either, actually passed) into `baseline_eval.json`, and appends each model tried to
`attempts.json`.

**Gate: accuracy >= 20% (`config.BASELINE_MIN_ACCURACY`) AND both graders' group
reward std > 0.** Step 2 below reads this and REFUSES to run (no GPU time spent) if
it's False. This cell ALSO sets `config.MODEL_NAME` in this notebook's own kernel to
match whichever model `baseline_eval.json` reports -- `baseline_eval.py` runs as a
separate subprocess, so its own internal model switch does not otherwise propagate
back here, and every later cell (determinism check, pilot gate) builds its model from
this kernel's `config.MODEL_NAME`.

In [ ]:
!python baseline_eval.py --out_dir "{DRIVE_RESULTS_DIR}"

import json, os
with open(os.path.join(DRIVE_RESULTS_DIR, "baseline_eval.json")) as f:
    baseline_result = json.load(f)
print(baseline_result)

config.MODEL_NAME = baseline_result['model_name']
print(f"\nconfig.MODEL_NAME set to {config.MODEL_NAME} for all later cells")

## 7. Determinism check

Two SEPARATE `run_one()` calls with the SAME method+seed (1 step each) should produce
(nearly) identical first-step advantages -- confirming seeding (LoRA init, GSM8K
grader assignment, generation sampling) is reproducible end to end in this new task,
before trusting the single-seed pilot gate below.

In [ ]:
import sys
sys.path.insert(0, '.')
from check_determinism import run_check

determinism_passed, determinism_rel_diff = run_check(method='grpo', seed=0)

## 8. STEP 2: PILOT GATE (1 seed) -- STOPS HERE

**Refuses to run at all if Step 1's `gate_passed` was False** (prints why and stops --
no GPU time spent). Otherwise runs Dr.GRPO and GRPO (1 seed each, using whichever
model Step 1 selected) on the real GSM8K conflicting-grader mix.

**PRE-REGISTERED gate, kept exactly as specified: PASS iff GRPO's final mean
completion length <= `config.PILOT_GATE_MAX_GRPO_TO_DRGRPO_LENGTH_RATIO` (0.85) x
Dr.GRPO's.** Dr.GRPO lacks GRPO's per-group reward normalization and is predicted to
be pulled toward the verbose (0-10) grader, so it should end up noticeably longer.
Prints PASS/FAIL with completion length, accuracy, AND both graders' mean reward for
BOTH runs, plus wall-clock per run; on FAIL, also prints diagnostics (did length move
from baseline at all? did accuracy move at all?) using Step 1's numbers. Every call
(refused or completed) is appended to `attempts.json`.

**This notebook stops here.** No further cells -- extending to a full
multi-seed sweep is a separate, later step once the pilot signal looks right.

In [ ]:
from pilot_gate import run_pilot_gate

pilot_summary = run_pilot_gate(seed=0, out_dir=DRIVE_RESULTS_DIR, skip_existing=True)